# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

In [2]:
"""The following generates the Quiz all our models will be evaluated on."""
import itertools
import string
import numpy as np

from dotenv import load_dotenv

load_dotenv("keys.env", verbose=True)

from smolbench.induction.chromatic import (
	ChromaticIntervalsConfig,
	Prompter,
	get_random_exclusive_quiz,
	anneal_intervals
)
from smolbench.evals import (
	Marks
)
from smolbench.evals.openrouter import (
	evaluate
)

template = string.Template(
	"You are a Boolean classifier.\n"
	"\n"
	"Task: determine whether the statement in the Question is logically "
	"possible given the Context.\n"
	"\n"
	"Output format:\n"
	"Return exactly one of these two strings and nothing else:\n"
	"True\n"
	"False\n"
	"\n"
	"Do not output any explanation, punctuation, quotes, labels, code fences, "
	"or extra whitespace."
	"Stop immediately after writing True or False."
	"\n"
	"Context:\n"
	"There is a ceremonial role called the $role, whose job it is to"
	" head the $parade parade. No one else besides the $role is able to head"
	" the $parade parade. The following lists the people who were $role and"
	" the years they were $role:\n"
	"$positive_info\n"
	"\n"
	"Question:\n"
	"Between the years $start and $end, exclusive of the end, could $color"
	" have headed the $parade parade every year?"
)

def query_gen(
	labels_to_intervals: Dict[Color, Intervals],
	interval_to_label: Dict[Intervals, Color],
	seed: int,
) -> Dict[str, str]:
	"""Generates a series of queries"""
	rng: np.random.Generator = np.random.default_rng(seed)
	# Finds max interval.
	n: int = max(interval[1] for interval in interval_to_label)
	for color, intervals in labels_to_intervals.items():
		# Generates a series of true items.
		for start, end in intervals:
			start, end = np.sort(
				rng.choice(range(start, end + 1), size=2, replace=False)
			)
			yield {"color": color, "start": start, "end": end}, True
		# Generates a false proposition.
		invalid_range: Intervals = anneal_intervals(
			itertools.chain(
				(set(interval_to_label.keys()) - set(itertools.chain(*intervals)))
			)
		)
		for start, end in invalid_range:
			start = rng.choice(range(start, end))
			# Binom with p = intervals / n capped at end for a similar-ish
			# distr. to positive accounts.
			end = min(
				end,
				start
				+ rng.binomial(
					end - start + 1,
					np.mean([len(interval) for interval in interval_to_label]) / n,
				)
				+ 1,
			)
			yield {"color": color, "start": start, "end": end}, False


SEED: int = 1776
intens_quiz, extens_quiz = get_random_exclusive_quiz(
	ChromaticIntervalsConfig(
            n=250,
            intervals=250 // 4,
            colors=45,
            seed=SEED,
        ),
	Prompter(
		template,
		{
			"role": "Twislax",
			"parade": "Gildane",
		},
		query_gen,
	),
)

## Prompt Validation

In [3]:
print(intens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. The following lists the people who were Twislax and the years they were Twislax:
Sl was Twislax on 24 to 34.
Wd was Twislax on 108 to 120 and 137 to 139.
vS was Twislax on 35 to 36 and 134 to 135.
Im was Twislax on 63 to 63.
xe was Twislax on 186 to 190.
zI was Twislax on 232 to 233.
Je was Twislax on 88 to 90.
Cj was Twislax on 38 to 41 and 201 to 204.
ld was Twislax on 0 to 10, 180 to 181, and 225 to 226.
Mw was Twislax on 61 to 62, 149 to 160, and 191 to 200.
PD was Twislax

In [4]:
print(extens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. The following lists the people who were Twislax and the years they were Twislax:
Sl was Twislax on 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, and 34.
Wd was Twislax on 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 137, 138, and 139.
vS was Twislax on 35, 36, 134, and 135.
Im was Twislax on 63.
xe was Twislax on 186, 187, 188, 189, and 190.
zI was Twislax on 232 and 233.
Je was Twislax on 88, 89, and 90.
Cj was Twislax on 38, 39, 40, 41, 201, 202, 203, and 204.


# Decoder-Only Model
This section tests classical decoder-only models.

In [5]:
decode_intens_eval: Marks = evaluate(intens_quiz, "mistralai/mistral-7b-instruct-v0.1", SEED)

0 0 0
0 1 0
' False. Sl was Twislax between the years 24 to 34, which is outside the range specified in the question.' is not a bool.
0 1 1
0 2 1
0 3 1
1 3 1
1 4 1
1 5 1
2 5 1
2 6 1
3 6 1
3 7 1
4 7 1
4 8 1
5 8 1
5 9 1
5 10 1
' False. Je was Twislax only in the years 88 to 90.' is not a bool.
5 10 2
5 11 2
5 12 2
6 12 2
6 13 2
6 14 2
6 15 2
7 15 2
7 16 2
7 17 2
7 18 2
8 18 2
9 18 2
9 19 2
10 19 2
10 20 2
10 21 2
11 21 2
' False.

The context states that yx was not Twislax during the years 166 to 169.' is not a bool.
11 21 3
' False.

According to the context, wE was not Twislax during the years 89 and 90.' is not a bool.
11 21 4
12 21 4
13 21 4
13 22 4
14 22 4
14 23 4
14 24 4
14 25 4
14 26 4
14 27 4
15 27 4
15 28 4
16 28 4
16 29 4
' False. Vp was Twislax only in the year 121.' is not a bool.
16 29 5
16 30 5
16 31 5
17 31 5
17 32 5
17 33 5
18 33 5
18 34 5
19 34 5
19 35 5
20 35 5
' False.

The context states that oF was not Twislax during the years 40 to 42.' is not a bool.
20 35 6
20 36 

In [6]:
decode_extens_eval: Marks = evaluate(extens_quiz, "mistralai/mistral-7b-instruct-v0.1", SEED)

0 0 0
0 1 0
1 1 0
1 2 0
1 3 0
1 4 0
1 5 0
2 5 0
3 5 0
3 6 0
4 6 0
4 7 0
5 7 0
5 8 0
6 8 0
7 8 0
7 9 0
8 9 0
9 9 0
9 10 0
9 11 0
9 12 0
9 13 0
' True

(ld was Twislax in the years 225, 226, and 180, which is before 227)' is not a bool.
9 13 1
10 13 1
11 13 1
12 13 1
' True

(Mw was Twislax in the years 191, 192, 193, 194, ..., 200)' is not a bool.
12 13 2
' True

(Mw was Twislax in the years 15, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 191, 192, 193, 194, 195, 196, 197, 198, 199, and 200. The years between 10 and 13, exclusive of the end, are 11, 12, and 13. Mw was Twislax in years 15, 151, and 152 which are before years 11, 12, and 13. However, Mw was also Twislax in years 153, 154, 155, 156, 157, 158, 159, 160, 191, 192, 193, 194, 195, 196, 197, 198, 199, and 200 which are after years 11, 12, and 13. Since the context states that no one else besides the Twislax is able to head the Gildane parade, it is logically possible that Mw headed the Gildane parade every year between 10

In [7]:
print(decode_intens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid)
print(decode_extens_eval.correct, decode_extens_eval.incorrect, decode_extens_eval.invalid)

36 60 11
54 41 12


## CoT Model
This section tests a CoT model.

In [8]:
cot_intens_eval: Marks = evaluate(intens_quiz, "deepseek/deepseek-r1-distill-qwen-32b", SEED)

OpenRouter request failed on attempt 1: Expecting value: line 39 column 1 (char 209). Retrying in 60 seconds.
OpenRouter request failed on attempt 1: Expecting value: line 67 column 1 (char 363). Retrying in 60 seconds.
0 0 0
1 0 0
2 0 0
3 0 0
4 0 0
4 1 0
5 1 0
6 1 0
7 1 0
8 1 0
9 1 0
10 1 0
11 1 0
12 1 0
13 1 0
14 1 0
15 1 0
16 1 0
17 1 0
18 1 0
19 1 0
20 1 0
21 1 0
22 1 0
23 1 0
24 1 0
25 1 0
26 1 0
27 1 0
28 1 0
29 1 0
30 1 0
31 1 0
32 1 0
33 1 0
34 1 0
35 1 0
36 1 0
'False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False

False


In [10]:
cot_extens_eval: Marks = evaluate(extens_quiz, "deepseek/deepseek-r1-distill-qwen-32b", SEED)

OpenRouter request failed on attempt 1: Expecting value: line 159 column 1 (char 869). Retrying in 60 seconds.
OpenRouter request failed on attempt 2: Expecting value: line 121 column 1 (char 660). Retrying in 60 seconds.
OpenRouter request failed on attempt 1: HTTPSConnectionPool(host='openrouter.ai', port=443): Read timed out. (read timeout=120). Retrying in 60 seconds.
0 0 0
'True

The years between 24 and 25 exclusive do not exist, so the condition is trivially true.

True' is not a bool.
0 0 1
1 0 1
2 0 1
'False

Wait, no, wait. Upon re-reading, the years listed for Wd include 137, 138, etc. The question is about between 137 and 138, exclusive. Since there are no years between them, it's implying whether Wd could have been the Twislax during that time, but since there are no years, it's a bit confusing.

But "exclusive of the end" means the period is 137 < year < 138. Since years are integers, there's no such year. Therefore, it's impossible for Wd to have headed the parade in a n

In [11]:
print(cot_intens_eval.correct, cot_intens_eval.incorrect, cot_intens_eval.invalid)
print(cot_extens_eval.correct, cot_extens_eval.incorrect, cot_extens_eval.invalid)

102 3 2
101 2 4
